# Dataset Exploration & Annotation Documentation
## Early Warning System for Forest Fire — Group 16
**CS437/CS5317/EE414/EE513 — Deep Learning, Spring 2026**  
Nausherwan Khan Arzani (27100384) · Saad Jamshaid Khan (27100159)

---

This notebook documents every dataset used across the three stages of our training pipeline:

| Stage | Purpose | Datasets |
|---|---|---|
| **Stage 1 — Pretraining** | Large-scale daytime fire/smoke detection | D-Fire, FIgLib, SKLFS-WildFire, FASDD, VisiFire, FIRESENSE |
| **Stage 2 — Nighttime Fine-tuning** | Low-light and IR-based fire detection | FLAME 2 (RGB + IR) |
| **Stage 3 — Domain Adaptation** | Site-specific PTZ camera footage | WWF Custom Dataset (annotation in progress) |

For each dataset we cover: origin and access, number of classes, train/val/test splits, sample images, annotation format, and relevance to our research question.

## 0. Setup & Dependencies

In [1]:
!pip3 install seaborn opencv-python pillow roboflow ultralytics

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [2]:
import sys
!{sys.executable} -m pip install seaborn opencv-python pillow roboflow ultralytics

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [3]:
# Install required packages
!pip3 install roboflow opencv-python matplotlib seaborn pandas numpy Pillow tqdm --quiet

import os
import json
import glob
import random
import urllib.request
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
from PIL import Image
from tqdm import tqdm

# Consistent style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('Environment ready.')

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Environment ready.


---
## 1. Dataset Landscape Overview

The table below summarises all datasets used in this project. Together they cover RGB daytime, RGB nighttime, thermal/IR, fixed-camera, and aerial viewpoints — addressing each of the three core gaps identified in our SOA survey.

In [4]:
dataset_overview = pd.DataFrame([
    {
        'Dataset':    'D-Fire',
        'Size':       '21,527 images',
        'Modality':   'RGB',
        'Classes':    'fire, smoke',
        'Task':       'Detection (YOLO)',
        'Split':      '70 / 15 / 15',
        'Access':     'Roboflow / GitHub',
        'Stage':      'Pretraining',
        'Key role':   'Primary compact-model benchmark'
    },
    {
        'Dataset':    'FIgLib',
        'Size':       '24,800 images',
        'Modality':   'RGB',
        'Classes':    'smoke (binary)',
        'Task':       'Classification / Detection',
        'Split':      '315 sequences / 101 cameras',
        'Access':     'HPWREN (free registration)',
        'Stage':      'Pretraining',
        'Key role':   'Closest public analogue to PTZ setup'
    },
    {
        'Dataset':    'SKLFS-WildFire',
        'Size':       '350,000+ images',
        'Modality':   'RGB',
        'Classes':    'smoke (early-stage)',
        'Task':       'Detection',
        'Split':      'Train / Val',
        'Access':     'arXiv:2311.10116 (authors)',
        'Stage':      'Pretraining',
        'Key role':   'Largest early-stage wildfire dataset'
    },
    {
        'Dataset':    'FASDD',
        'Size':       '120,000+ images',
        'Modality':   'RGB',
        'Classes':    'fire, smoke',
        'Task':       'Detection',
        'Split':      'Train / Test',
        'Access':     'Public (GitHub)',
        'Stage':      'Pretraining',
        'Key role':   'Geographic diversity / generalisation testing'
    },
    {
        'Dataset':    'VisiFire',
        'Size':       '40 video clips',
        'Modality':   'RGB',
        'Classes':    'fire, smoke',
        'Task':       'Classification',
        'Split':      'No fixed split',
        'Access':     'Public (University of Bilkent)',
        'Stage':      'Pretraining',
        'Key role':   'Forest + urban scene variety'
    },
    {
        'Dataset':    'FIRESENSE',
        'Size':       '49 video clips',
        'Modality':   'RGB',
        'Classes':    'fire, smoke',
        'Task':       'Classification / Detection',
        'Split':      'No fixed split',
        'Access':     'Public (EU FP7 Project)',
        'Stage':      'Pretraining',
        'Key role':   'Heritage/urban hard negatives'
    },
    {
        'Dataset':    'FLAME 2',
        'Size':       '7 RGB + 7 IR videos',
        'Modality':   'RGB + Thermal IR',
        'Classes':    'fire (binary)',
        'Task':       'Classification / Segmentation',
        'Split':      '39,375 train / 8,617 test images',
        'Access':     'IEEE DataPort (free account)',
        'Stage':      'Nighttime Fine-tuning',
        'Key role':   'Only public paired RGB-IR dataset with wildfire'
    },
    {
        'Dataset':    'WWF PTZ (Custom)',
        'Size':       'TBD',
        'Modality':   'RGB',
        'Classes':    'fire, smoke, cloud (hard neg.)',
        'Task':       'Detection (YOLO)',
        'Split':      'TBD (hold out 1 camera site)',
        'Access':     'Internal — WWF Pakistan',
        'Stage':      'Domain Adaptation',
        'Key role':   'South Asian forest terrain; only dataset of its kind'
    },
])

pd.set_option('display.max_colwidth', 60)
dataset_overview.set_index('Dataset', inplace=True)
print('=== All Datasets — Project Pipeline ===')
dataset_overview

=== All Datasets — Project Pipeline ===


,Size,Modality,Classes,Task,Split,Access,Stage,Key role
Dataset,,,,,,,,
D-Fire,"21,527 images",RGB,"fire, smoke",Detection (YOLO),70 / 15 / 15,Roboflow / GitHub,Pretraining,Primary compact-model benchmark
FIgLib,"24,800 images",RGB,smoke (binary),Classification / Detection,315 sequences / 101 cameras,HPWREN (free registration),Pretraining,Closest public analogue to PTZ setup
SKLFS-WildFire,"350,000+ images",RGB,smoke (early-stage),Detection,Train / Val,arXiv:2311.10116 (authors),Pretraining,Largest early-stage wildfire dataset
FASDD,"120,000+ images",RGB,"fire, smoke",Detection,Train / Test,Public (GitHub),Pretraining,Geographic diversity / generalisation testing
VisiFire,40 video clips,RGB,"fire, smoke",Classification,No fixed split,Public (University of Bilkent),Pretraining,Forest + urban scene variety
FIRESENSE,49 video clips,RGB,"fire, smoke",Classification / Detection,No fixed split,Public (EU FP7 Project),Pretraining,Heritage/urban hard negatives
FLAME 2,7 RGB + 7 IR videos,RGB + Thermal IR,fire (binary),Classification / Segmentation,"39,375 train / 8,617 test images",IEEE DataPort (free account),Nighttime Fine-tuning,Only public paired RGB-IR dataset with wildfire
WWF PTZ (Custom),TBD,RGB,"fire, smoke, cloud (hard neg.)",Detection (YOLO),TBD (hold out 1 camera site),Internal — WWF Pakistan,Domain Adaptation,South Asian forest terrain; only dataset of its kind


---
## 2. Stage 1 — Pretraining Datasets

### 2.1 D-Fire (Primary Benchmark Dataset)

**Origin:** Collected by Pedro Vinícius A. B. de Venâncio et al., publicly released as part of the paper *"An automatic fire detection system based on deep convolutional neural networks"*. Available via Roboflow Universe and GitHub.

**Why this dataset:**  
D-Fire is the standard benchmark used to evaluate compact fire/smoke models in our SOA (referenced in the `uncertainty2025` paper, which benchmarks YOLOv5n and YOLOv8n on it). It has two independent object classes — `fire` and `smoke` — enabling us to measure the fire/smoke recall gap quantified in `yolov8_2024` (recall 0.9 vs 0.7).

**Classes:** 2 — `fire` (class 0), `smoke` (class 1)  
**Annotation format:** YOLO format (normalised `cx cy w h` per line in `.txt` files)

In [6]:
import shutil
from pathlib import Path

# archive-3 is the D-Fire download from Kaggle
src = Path('/Users/saadjamshaidkhan/LUMS/DeepLearning/project/archive-3')
dst = Path('/Users/saadjamshaidkhan/LUMS/DeepLearning/project/datasets/dfire')

# Check what's inside archive-3 first
for item in sorted(src.iterdir()):
    print(item.name)

.DS_Store
data
data.yaml


In [7]:
import shutil
from pathlib import Path

src = Path('/Users/saadjamshaidkhan/LUMS/DeepLearning/project/archive-3/data')
dst = Path('/Users/saadjamshaidkhan/LUMS/DeepLearning/project/datasets/dfire')

# Remove the empty dfire folder that's already there and replace with real data
if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)
print("D-Fire copied. Contents:")
for item in sorted(dst.iterdir()):
    print(f"  {item.name}/")

D-Fire copied. Contents:
  .DS_Store/
  test/
  train/
  val/


In [8]:
# Check what's inside the Wildfire Smoke folder
src = Path('/Users/saadjamshaidkhan/LUMS/DeepLearning/project/Wildfire Smoke')
for item in sorted(src.iterdir()):
    print(item.name)

.DS_Store
README.dataset.txt
README.roboflow.txt
data.yaml
test
train
valid


In [9]:
src = Path('/Users/saadjamshaidkhan/LUMS/DeepLearning/project/Wildfire Smoke')
dst = Path('/Users/saadjamshaidkhan/LUMS/DeepLearning/project/datasets/figlib')

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)
print("Wildfire Smoke copied. Contents:")
for item in sorted(dst.iterdir()):
    print(f"  {item.name}")

Wildfire Smoke copied. Contents:
  .DS_Store
  README.dataset.txt
  README.roboflow.txt
  data.yaml
  test
  train
  valid


In [10]:


from pathlib import Path

datasets = {
    'D-Fire':         Path('./datasets/dfire'),
    'Wildfire Smoke': Path('./datasets/figlib'),
}

for name, root in datasets.items():
    print(f"\n{name}:")
    for split in ['train', 'valid', 'val', 'test']:
        img_dir   = root / split / 'images'
        label_dir = root / split / 'labels'
        if img_dir.exists():
            n_img = len(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')))
            n_lbl = len(list(label_dir.glob('*.txt'))) if label_dir.exists() else 0
            print(f"  {split:6s}: {n_img:6,} images | {n_lbl:6,} labels")


D-Fire:
  train : 14,122 images | 14,122 labels
  val   :  3,099 images |  3,099 labels
  test  :  4,306 images |  4,306 labels

Wildfire Smoke:
  train :    516 images |    516 labels
  valid :    147 images |    147 labels
  test  :     74 images |     74 labels


In [5]:

# D-Fire is hosted directly on GitHub
# Clone the repository (it contains the full dataset)
!git clone https://github.com/gaia-solutions-on-demand/DFireDataset.git ./datasets/dfire

fatal: destination path './datasets/dfire' already exists and is not an empty directory.


In [ ]:
# # ── Define DFIRE_ROOT and verify folder structure ─────────────────────────────
# DFIRE_ROOT = Path('./datasets/dfire')

# print('Folder structure:')
# for item in sorted(DFIRE_ROOT.rglob('*')):
#     if item.is_dir():
#         n_files = len(list(item.glob('*')))
#         print(f'  {item.relative_to(DFIRE_ROOT)} / ({n_files} files)')

Folder structure:
  .git / (10 files)
  .git/hooks / (14 files)
  .git/info / (1 files)
  .git/logs / (2 files)
  .git/logs/refs / (2 files)
  .git/logs/refs/heads / (1 files)
  .git/logs/refs/remotes / (1 files)
  .git/logs/refs/remotes/origin / (1 files)
  .git/objects / (2 files)
  .git/objects/info / (0 files)
  .git/objects/pack / (3 files)
  .git/refs / (3 files)
  .git/refs/heads / (1 files)
  .git/refs/remotes / (1 files)
  .git/refs/remotes/origin / (1 files)
  .git/refs/tags / (0 files)
  figures / (1 files)
  utils / (1 files)


In [26]:
!brew install git-lfs          # one-time install on Mac
!git -C ./datasets/dfire lfs install
!git -C ./datasets/dfire lfs pull

To reinstall 3.7.1, run:
  brew reinstall git-lfs
Updated Git hooks.
Git LFS initialized.


In [ ]:
# # ── Download D-Fire via Roboflow ──────────────────────────────────────────────
# # Requires a free Roboflow account. Get your API key from:
# # https://app.roboflow.com → Settings → Roboflow API

# from roboflow import Roboflow

# RF_API_KEY = "GiTi4abnSjmmZkClXGXW"   # ← replace with your key

# rf = Roboflow(api_key=RF_API_KEY)
# project = rf.workspace("dataset-xrqna").project("d-fire")
# dataset = project.version(1).download("yolov8", location="./datasets/dfire")

# DFIRE_ROOT = Path('./datasets/dfire')
# print(f'D-Fire downloaded to: {DFIRE_ROOT}')

loading Roboflow workspace...


RoboflowError: {"error":{"message":"Unsupported get request. Workspace with ID \"dataset-xrqna\" does not exist or cannot be loaded due to missing permissions.","status":404,"type":"GraphMethodException","hint":"You can see your available workspaces by issuing a GET request to /workspaces"}}

In [ ]:
# !git -C ./datasets/dfire lfs pull

In [30]:
import os

# Set credentials directly as environment variables
os.environ['KAGGLE_USERNAME'] = 'merc11123saad'
os.environ['KAGGLE_KEY'] = 'KGAT_8ee63e66161acf22ca1650c901f18a3e'

# Download and unzip D-Fire
!pip3 install kaggle -q
!kaggle datasets download -d pedromorfeu/d-fire-dataset -p ./datasets/ --unzip

print("Done!")

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
/Users/saadjamshaidkhan/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
403 Client Error: Forbidden for url: https://www.kaggle.com/api/v1/datasets/metadata/pedromorfeu/d-fire-dataset
Done!


In [32]:
import os
from pathlib import Path

# Check what's in datasets folder
datasets_path = Path('./datasets')
print("Contents of ./datasets/:")
for item in sorted(datasets_path.iterdir()):
    size_mb = item.stat().st_size / 1e6 if item.is_file() else 0
    print(f"  {item.name}  {'(' + str(round(size_mb,1)) + ' MB)' if item.is_file() else '(dir)'}")

Contents of ./datasets/:
  dfire  (dir)


In [31]:
!pip3 install gdown
import gdown
import zipfile
from pathlib import Path

# Official D-Fire Google Drive link from the paper authors
gdown.download(
    "https://drive.google.com/uc?id=1pFPSMV9-MDF4QFjqXNGFYFHey3N0Rnx_",
    output="./datasets/dfire.zip",
    quiet=False
)

# Extract
with zipfile.ZipFile("./datasets/dfire.zip", 'r') as z:
    z.extractall("./datasets/dfire_extracted")

print("Done. Checking contents...")
import os
for root, dirs, files in os.walk("./datasets/dfire_extracted"):
    level = root.replace("./datasets/dfire_extracted", "").count(os.sep)
    if level < 3:
        print(f"{'  ' * level}{os.path.basename(root)}/ ({len(files)} files)")

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


FileURLRetrievalError: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1pFPSMV9-MDF4QFjqXNGFYFHey3N0Rnx_

but Gdown can't. Please check connections and permissions.

In [28]:
# ── D-Fire: Folder structure ──────────────────────────────────────────────────
def count_split(root, split):
    img_dir   = root / split / 'images'
    label_dir = root / split / 'labels'
    imgs   = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
    labels = list(label_dir.glob('*.txt'))
    return len(imgs), len(labels)

for split in ['train', 'valid', 'test']:
    n_img, n_lbl = count_split(DFIRE_ROOT, split)
    print(f'  {split:6s}: {n_img:6d} images | {n_lbl:6d} label files')

  train :      0 images |      0 label files
  valid :      0 images |      0 label files
  test  :      0 images |      0 label files


In [ ]:
# ── D-Fire: Parse all labels to count class instances ────────────────────────
CLASS_NAMES = {0: 'fire', 1: 'smoke'}

def parse_yolo_labels(root, splits=('train', 'valid', 'test')):
    records = []
    for split in splits:
        label_dir = root / split / 'labels'
        for lbl_path in label_dir.glob('*.txt'):
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        cls, cx, cy, w, h = int(parts[0]), *map(float, parts[1:])
                        records.append({'split': split, 'class_id': cls,
                                        'cx': cx, 'cy': cy, 'bw': w, 'bh': h,
                                        'area': w * h})
    return pd.DataFrame(records)

df_dfire = parse_yolo_labels(DFIRE_ROOT)
df_dfire['class_name'] = df_dfire['class_id'].map(CLASS_NAMES)
print(f'Total bounding box annotations: {len(df_dfire):,}')
print(df_dfire.groupby(['split', 'class_name']).size().unstack(fill_value=0))

In [ ]:
# ── D-Fire: Class distribution chart ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Per-class totals
class_counts = df_dfire['class_name'].value_counts()
axes[0].bar(class_counts.index, class_counts.values,
            color=['#e74c3c', '#95a5a6'], edgecolor='black', linewidth=0.7)
axes[0].set_title('D-Fire: Total Annotations by Class', fontweight='bold')
axes[0].set_ylabel('Bounding Box Count')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontsize=10)

# Per-split breakdown
split_class = df_dfire.groupby(['split', 'class_name']).size().unstack(fill_value=0)
split_class.plot(kind='bar', ax=axes[1], color=['#e74c3c', '#95a5a6'],
                 edgecolor='black', linewidth=0.7)
axes[1].set_title('D-Fire: Class Distribution per Split', fontweight='bold')
axes[1].set_ylabel('Bounding Box Count')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Class')

plt.tight_layout()
plt.savefig('dfire_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── D-Fire: Sample annotated images ──────────────────────────────────────────
def draw_yolo_boxes(img_path, label_path, class_names, colors):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if label_path.exists():
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls_id = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:])
                x1 = int((cx - bw/2) * w)
                y1 = int((cy - bh/2) * h)
                x2 = int((cx + bw/2) * w)
                y2 = int((cy + bh/2) * h)
                color = colors.get(cls_id, (255, 255, 0))
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                label = class_names.get(cls_id, str(cls_id))
                cv2.putText(img, label, (x1, y1 - 6),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
    return img

COLORS = {0: (231, 76, 60), 1: (149, 165, 166)}  # fire=red, smoke=grey

train_imgs = sorted((DFIRE_ROOT / 'train' / 'images').glob('*.jpg'))
sample_imgs = random.sample(train_imgs, min(8, len(train_imgs)))

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle('D-Fire — Sample Training Images with YOLO Annotations', fontsize=13, fontweight='bold')

for ax, img_path in zip(axes.flat, sample_imgs):
    lbl_path = DFIRE_ROOT / 'train' / 'labels' / (img_path.stem + '.txt')
    vis = draw_yolo_boxes(img_path, lbl_path, CLASS_NAMES, COLORS)
    ax.imshow(vis)
    ax.axis('off')
    ax.set_title(img_path.name[:20], fontsize=7)

plt.tight_layout()
plt.savefig('dfire_samples.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── D-Fire: Image statistics (resolution, brightness) ────────────────────────
def compute_image_stats(img_dir, n_sample=300):
    paths = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
    paths = random.sample(paths, min(n_sample, len(paths)))
    stats = []
    for p in tqdm(paths, desc=str(img_dir.parent.name)):
        img = cv2.imread(str(p), cv2.IMREAD_COLOR)
        if img is None:
            continue
        h, w = img.shape[:2]
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        stats.append({'width': w, 'height': h,
                      'aspect': round(w/h, 2),
                      'brightness': gray.mean()})
    return pd.DataFrame(stats)

stats_train = compute_image_stats(DFIRE_ROOT / 'train' / 'images')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(stats_train['width'],  bins=20, color='steelblue', edgecolor='black')
axes[0].set_title('Image Width Distribution');  axes[0].set_xlabel('Pixels')
axes[1].hist(stats_train['height'], bins=20, color='steelblue', edgecolor='black')
axes[1].set_title('Image Height Distribution'); axes[1].set_xlabel('Pixels')
axes[2].hist(stats_train['brightness'], bins=20, color='#e67e22', edgecolor='black')
axes[2].set_title('Mean Brightness Distribution'); axes[2].set_xlabel('Pixel Value (0–255)')

for ax in axes:
    ax.set_ylabel('Count')

fig.suptitle('D-Fire Training Set — Image Statistics', fontweight='bold')
plt.tight_layout()
plt.savefig('dfire_stats.png', dpi=150, bbox_inches='tight')
plt.show()

print(stats_train.describe().round(1))

In [ ]:
# ── D-Fire: Bounding box size distribution ────────────────────────────────────
# Box area is normalised (0–1); small values = small targets (early-stage smoke)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for cls_name, grp in df_dfire[df_dfire['split'] == 'train'].groupby('class_name'):
    axes[0].hist(grp['area'], bins=40, alpha=0.6, label=cls_name)
axes[0].set_title('Bounding Box Area (normalised) — Train Split')
axes[0].set_xlabel('Box Area (cx*cy normalised)')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].set_xlim(0, 0.5)

df_dfire['bw_px_est'] = df_dfire['bw'] * 640  # approximate at 640px
df_dfire['bh_px_est'] = df_dfire['bh'] * 640
axes[1].scatter(df_dfire['bw_px_est'], df_dfire['bh_px_est'],
                c=df_dfire['class_id'].map({0:'#e74c3c', 1:'#95a5a6'}),
                alpha=0.3, s=5)
axes[1].set_title('Box Width vs Height (est. at 640px)')
axes[1].set_xlabel('Box Width (px)')
axes[1].set_ylabel('Box Height (px)')
import matplotlib.patches as mpatches
axes[1].legend(handles=[
    mpatches.Patch(color='#e74c3c', label='fire'),
    mpatches.Patch(color='#95a5a6', label='smoke')
])

plt.suptitle('D-Fire — Bounding Box Size Analysis', fontweight='bold')
plt.tight_layout()
plt.savefig('dfire_bbox_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── D-Fire: Annotation format walkthrough ─────────────────────────────────────
# Show one raw label file so the reader understands the format
sample_lbl = list((DFIRE_ROOT / 'train' / 'labels').glob('*.txt'))[0]

print('=== Raw YOLO label file:', sample_lbl.name, '===')
with open(sample_lbl) as f:
    raw = f.read()
print(raw)

print()
print('Format: <class_id>  <cx>  <cy>  <width>  <height>')
print('All values are normalised to [0, 1] relative to image dimensions.')
print('class_id 0 = fire  |  class_id 1 = smoke')

# Decode one box as example
parts = raw.strip().split('\n')[0].split()
cls, cx, cy, bw, bh = int(parts[0]), *map(float, parts[1:])
IMG_W, IMG_H = 640, 640  # assume standard
x1 = (cx - bw/2) * IMG_W
y1 = (cy - bh/2) * IMG_H
x2 = (cx + bw/2) * IMG_W
y2 = (cy + bh/2) * IMG_H
print(f'\nDecoded example: class={CLASS_NAMES[cls]}, '
      f'box=({x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f}) px at 640×640')

---
### 2.2 FIgLib — Fixed-View Camera Dataset (Closest PTZ Analogue)

**Origin:** Dewangan et al., *Remote Sensing* 2022. Collected from the HPWREN camera network in Southern California — 101 fixed-view cameras across 315 wildland fire sequences.  
**Access:** Free registration at [hpwren.ucsd.edu/HPWREN-FIgLib](http://hpwren.ucsd.edu/HPWREN-FIgLib/)  

**Why this dataset:**  
FIgLib is the single public dataset most architecturally similar to our PTZ deployment. Cameras are fixed at elevated positions looking across horizons — exactly the geometry where cloud/smoke confusion is worst. The SmokeyNet model trained on FIgLib uses spatiotemporal aggregation across consecutive frames, which is our key architectural reference.

**Classes:** Binary — `smoke` / `no smoke` (image-level label per sequence frame)  
**Size:** 24,800 high-resolution images  
**Split:** By fire event (not by random image) — ensures geographic non-leakage

In [ ]:
# ── FIgLib: Download (requires HPWREN registration) ───────────────────────────
# After registering at http://hpwren.ucsd.edu/HPWREN-FIgLib/
# you will receive a download link. Place the extracted folder at:
FIGLIB_ROOT = Path('./datasets/figlib')

# Alternatively, use the SmokeyNet preprocessed split:
# https://github.com/aiformankind/wildfire-smoke-dataset

# ── Structural summary (update counts after download) ─────────────────────────
figlib_summary = pd.DataFrame([
    {'Split': 'Train', 'Fire sequences': 220, 'Images': '~17,400', 'Smoke frames': '~8,700'},
    {'Split': 'Val',   'Fire sequences':  47, 'Images':  '~3,700', 'Smoke frames': '~1,850'},
    {'Split': 'Test',  'Fire sequences':  48, 'Images':  '~3,700', 'Smoke frames': '~1,850'},
])
print('FIgLib Dataset — Split Summary (from published paper)')
print(figlib_summary.to_string(index=False))
print()
print('Note: Splits are by fire EVENT, not random image shuffle.')
print('This prevents the same camera view appearing in both train and test,')  
print('which would inflate generalisation metrics.')
print()
print('Class balance: ~50/50 smoke vs no-smoke (by design).')
print('Real deployment ratio is ~1 fire frame per ~10,000 non-fire frames.')

In [ ]:
# ── FIgLib: If downloaded, visualise sample frames ───────────────────────────
if FIGLIB_ROOT.exists():
    smoke_imgs    = list((FIGLIB_ROOT / 'train' / 'smoke').glob('*.jpg'))[:4]
    no_smoke_imgs = list((FIGLIB_ROOT / 'train' / 'no_smoke').glob('*.jpg'))[:4]

    fig, axes = plt.subplots(2, 4, figsize=(16, 6))
    fig.suptitle('FIgLib — Fixed-Camera Frames (Top: Smoke / Bottom: No Smoke)',
                 fontsize=12, fontweight='bold')
    for ax, p in zip(axes[0], smoke_imgs):
        ax.imshow(np.array(Image.open(p)))
        ax.set_title('SMOKE', color='red', fontsize=9)
        ax.axis('off')
    for ax, p in zip(axes[1], no_smoke_imgs):
        ax.imshow(np.array(Image.open(p)))
        ax.set_title('NO SMOKE', color='green', fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig('figlib_samples.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('FIgLib not yet downloaded.')
    print('Register at: http://hpwren.ucsd.edu/HPWREN-FIgLib/')
    print('Expected path:', FIGLIB_ROOT)

In [ ]:
# ── FIgLib: Key properties important for our pipeline ────────────────────────
figlib_properties = {
    'Camera type':           'Fixed-view (pan-tilt-zoom analogous)',
    'Resolution':            '~1632 × 1224 px (high resolution)',
    'Geography':             'Southern California — chaparral and woodland',
    'Temporal structure':    'Sequential frames — enables spatiotemporal modelling',
    'Hard negatives':        'Morning haze and fog naturally present in horizon shots',
    'Annotation format':     'Image-level binary label per frame',
    'Gap vs our project':    'No South Asian terrain; no bounding boxes for detection',
    'Usage in our pipeline': 'Pre-train temporal smoke classifier before PTZ fine-tuning',
}
for k, v in figlib_properties.items():
    print(f'  {k:<25}: {v}')

---
### 2.3 SKLFS-WildFire — Early-Stage Wildfire Dataset

**Origin:** Introduced alongside the *Wildfire Smoke Detection System* paper (arXiv:2311.10116). Specifically designed to capture fires at the earliest visible stage — when smoke column is narrow and barely distinguishable.  
**Access:** Contact authors via arXiv:2311.10116 (not yet on Roboflow/GitHub).

**Why this dataset:**  
At 350,000+ images this is the largest single real early-stage wildfire dataset. The paper demonstrates that standard transformers fail on early smoke because low-level texture features are discarded — this motivates the CCPE module we benchmark against. For us it is a supplementary pretraining source.

**Classes:** `smoke` (focus is on pre-flame early smoke column)  

In [ ]:
# ── SKLFS: Dataset summary (from paper — no local download required) ──────────
sklfs_summary = pd.DataFrame([
    {'Subset':       'SKLFS-WildFire Train',
     'Images':       '299,827',
     'Source':       'Real wildfire surveillance cameras + web crawl',
     'Smoke type':   'Early-stage column smoke',
     'Hard neg.':    'Cloud, haze, dust included as negatives'},
    {'Subset':       'SKLFS-WildFire Val',
     'Images':       '~50,000',
     'Source':       'Same pipeline',
     'Smoke type':   'Early-stage column smoke',
     'Hard neg.':    'Cloud, haze, dust included as negatives'},
])
print(sklfs_summary.to_string(index=False))
print()
print('Key insight: 350K+ images still insufficient to solve early smoke.')
print('Authors propose Cross Contrast Patch Embedding (CCPE) to capture')
print('subtle low-level texture differences invisible to standard ViT patches.')

---
### 2.4 FASDD — Fire and Smoke Detection Dataset

**Origin:** Publicly released; 120,000+ heterogeneous images from diverse geographic locations and environmental conditions.  
**Access:** GitHub (public)

**Why this dataset:**  
FASDD is used for generalisation testing in our pipeline because of its geographic diversity. Our SOA notes that most public datasets are geographically biased toward North America and Europe. FASDD partially addresses this and serves as an out-of-distribution evaluation set.

In [ ]:
# ── FASDD: Summary ────────────────────────────────────────────────────────────
fasdd_summary = {
    'Total images':       '120,000+',
    'Classes':            'fire, smoke (bounding box)',
    'Annotation format':  'YOLO format',
    'Scene diversity':    'Forest, urban, industrial, vehicles, indoor',
    'Geographic scope':   'Multi-continent (broadest of all public datasets)',
    'Splits':             'Train / Test (no fixed validation split)',
    'Usage in pipeline':  'OOD generalisation testing after PTZ fine-tuning',
    'Limitation':         'Label quality inconsistent across geographic subsets',
    'Access URL':         'https://github.com/kenshiro-o/fire-detection-yolov5',
}
for k, v in fasdd_summary.items():
    print(f'  {k:<25}: {v}')

---
### 2.5 VisiFire & FIRESENSE — Video-Based Datasets

These two datasets provide the video/temporal dimension that static image datasets lack. They are used to verify that temporal models generalise across scene types.

In [ ]:
# ── VisiFire & FIRESENSE: Comparison ─────────────────────────────────────────
video_datasets = pd.DataFrame([
    {
        'Dataset':         'VisiFire',
        'Clips':           40,
        'Approx. frames':  '~10,000',
        'Classes':         'fire, smoke',
        'Scene types':     'Forest, field, urban',
        'Annotation':      'Frame-level binary',
        'Access':          'visfire.cs.bilkent.edu.tr',
        'Limitation':      'Small scale; no bounding boxes',
    },
    {
        'Dataset':         'FIRESENSE',
        'Clips':           49,
        'Approx. frames':  '~25,000',
        'Classes':         'fire, smoke',
        'Scene types':     'Heritage buildings, urban, indoor',
        'Annotation':      'Frame-level + region boxes',
        'Access':          'mklab.iti.gr/firesense',
        'Limitation':      'Urban bias; few forest scenes',
    },
])
video_datasets.set_index('Dataset', inplace=True)
print('Video Dataset Comparison')
video_datasets.T

---
## 3. Stage 2 — Nighttime Fine-tuning: FLAME 2

**Origin:** Introduced in *FLAME 2: FIRE Detection and Modeling* — IEEE DataPort. Collected using DJI Phantom and Matrice UAVs equipped with both RGB and FLIR thermal cameras during controlled wildfire experiments.

**Why this dataset:**  
FLAME 2 is the only publicly available dataset with paired RGB + IR sequences from actual wildfire events. Our SOA identifies nighttime detection as severely underrepresented. FLAME 2 is therefore used in Stage 2 of our pipeline to fine-tune the model on heat-signature-based detection before synthetic nighttime augmentation is applied to WWF footage.

**Classes:** Binary — `fire` / `no fire` (image-level), plus segmentation masks  
**Access:** [IEEE DataPort](https://ieee-dataport.org/open-access/flame-2-fire-detection-and-modeling-dataset) (free IEEE account)

In [ ]:
# ── FLAME 2: Structure ────────────────────────────────────────────────────────
flame2 = pd.DataFrame([
    {'Repository': '1 — RGB Video 1 (Zenmuse X4S)',    'Duration': '16 min', 'FPS': 29, 'Res': '1280×720',  'Modality': 'RGB'},
    {'Repository': '2 — RGB Video 2 (Zenmuse X4S)',    'Duration': '16 min', 'FPS': 29, 'Res': '1280×720',  'Modality': 'RGB'},
    {'Repository': '3 — IR WhiteHot',                  'Duration': '89 s',   'FPS': 30, 'Res': '640×512',   'Modality': 'Thermal IR'},
    {'Repository': '4 — IR GreenHot',                  'Duration': '5 min',  'FPS': 30, 'Res': '640×512',   'Modality': 'Thermal IR'},
    {'Repository': '5 — IR Fusion Heatmap',            'Duration': '25 min', 'FPS': 30, 'Res': '640×512',   'Modality': 'Thermal IR'},
    {'Repository': '6 — 4K RGB (DJI Phantom 3)',       'Duration': '17 min', 'FPS': 30, 'Res': '3840×2160', 'Modality': 'RGB'},
    {'Repository': '7 — Classification images (train)','Duration': 'N/A',    'FPS': 'N/A', 'Res': '254×254', 'Modality': 'RGB'},
    {'Repository': '8 — Classification images (test)', 'Duration': 'N/A',    'FPS': 'N/A', 'Res': '254×254', 'Modality': 'RGB'},
    {'Repository': '9 — Segmentation images',          'Duration': 'N/A',    'FPS': 'N/A', 'Res': 'High-res','Modality': 'RGB'},
    {'Repository': '10 — Segmentation masks',          'Duration': 'N/A',    'FPS': 'N/A', 'Res': 'High-res','Modality': 'Binary mask'},
])
print('FLAME 2 — Repository Structure (from paper)')
flame2

In [ ]:
# ── FLAME 2: Classification split ─────────────────────────────────────────────
flame2_clf = pd.DataFrame([
    {'Split': 'Train', 'Fire images': 23,022,  'No-fire images': 16,353, 'Total': 39375},
    {'Split': 'Test',  'Fire images':  8,617,  'No-fire images':  0,     'Total': 8617},
])
print('FLAME 2 — Classification Task Split (repositories 7 & 8)')
print(flame2_clf.to_string(index=False))
print()
print('Note: test set contains ONLY fire images — evaluation is recall-focused.')
print()
print('Usage in our pipeline:')
print('  - RGB sequences → fine-tune daytime model on UAV fire footage')
print('  - IR sequences  → train IR-aware head for nighttime glow detection')
print('  - Segmentation  → masks can be converted to YOLO detection boxes')

# ── FLAME 2: Synthetic nighttime augmentation plan ────────────────────────────
print()
print('Synthetic Nighttime Augmentation (applied to FLAME 2 RGB + D-Fire frames):')
aug_steps = [
    ('1. Darkening',         'Multiply pixel values by 0.1–0.3 to simulate night exposure'),
    ('2. Moonlight tint',    'Add slight blue channel bias (simulate moonlight colour temperature)'),
    ('3. Sensor noise',      'Add Gaussian noise σ=10–25 (simulate night camera noise floor)'),
    ('4. Fire preservation', 'Fire-region pixels restored at 60–80% original brightness'),
    ('5. CLAHE',             'Apply Contrast Limited AHE to simulate IR-like local contrast boost'),
]
for step, desc in aug_steps:
    print(f'  {step:<25}: {desc}')

In [ ]:
# ── FLAME 2: Visualise synthetic nighttime augmentation ───────────────────────
# Demonstrate the augmentation pipeline on any D-Fire image (no FLAME 2 needed)

def apply_nighttime_augmentation(img_bgr, seed=None):
    """Synthetic nighttime augmentation for fire/smoke images."""
    rng = np.random.default_rng(seed)
    img = img_bgr.astype(np.float32)

    # 1. Detect fire-like region (high red, low blue) before darkening
    fire_mask = (img[:,:,2] > 150) & (img[:,:,0] < 100)  # BGR: high R, low B

    # 2. Global darkening
    dark_factor = rng.uniform(0.12, 0.28)
    img = img * dark_factor

    # 3. Moonlight blue tint
    img[:,:,0] = np.clip(img[:,:,0] * 1.15, 0, 255)  # boost Blue channel

    # 4. Gaussian noise
    noise = rng.normal(0, 12, img.shape)
    img = np.clip(img + noise, 0, 255)

    # 5. Restore fire regions partially
    restore = rng.uniform(0.55, 0.75)
    img[fire_mask] = img_bgr.astype(np.float32)[fire_mask] * restore

    # 6. CLAHE on luminance channel
    img_u8 = img.astype(np.uint8)
    lab = cv2.cvtColor(img_u8, cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    lab[:,:,0] = clahe.apply(lab[:,:,0])
    img_u8 = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

    return img_u8


# Apply to sample D-Fire images
sample_paths = random.sample(train_imgs, 4)

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle('Synthetic Nighttime Augmentation — Applied to D-Fire Training Images',
             fontsize=12, fontweight='bold')

for i, img_path in enumerate(sample_paths):
    orig_bgr = cv2.imread(str(img_path))
    aug_bgr  = apply_nighttime_augmentation(orig_bgr, seed=i)
    axes[0, i].imshow(cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB))
    axes[0, i].set_title(f'Original', fontsize=9)
    axes[0, i].axis('off')
    axes[1, i].imshow(cv2.cvtColor(aug_bgr,  cv2.COLOR_BGR2RGB))
    axes[1, i].set_title(f'Augmented (Night)', fontsize=9, color='navy')
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('nighttime_augmentation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Stage 3 — Domain Adaptation: WWF Custom Dataset

The WWF PTZ camera footage represents the most important and most under-resourced dataset in this project. No public dataset covers South Asian forest terrain filmed from fixed elevated PTZ cameras. This section documents our annotation plan and the effort invested in building this dataset.

### 4.1 Why a Custom Dataset is Necessary

| Factor | Public datasets | WWF PTZ dataset |
|---|---|---|
| Geography | N. America / Europe | South Asia (Pakistan forests) |
| Camera type | UAV / handheld | Fixed PTZ (horizon-facing) |
| Time of day | Predominantly daytime | Daytime + dawn/dusk + nighttime |
| Cloud confounders | Rarely annotated | Explicitly annotated as hard negatives |
| Forest type | Chaparral / boreal | Sub-tropical mixed forest |

In [ ]:
# ── WWF Dataset: Annotation plan ─────────────────────────────────────────────
wwf_plan = {
    'Annotation tool':       'Roboflow Annotate (web-based, YOLO export)',
    'Annotation format':     'YOLO format — class cx cy w h (normalised)',
    'Classes': {
        0: 'fire   — visible flame region',
        1: 'smoke  — smoke plume or column',
        2: 'cloud  — hard negative: cloud resembling smoke on horizon',
    },
    'Inter-annotator agreement': 'Two annotators per image; IoU > 0.5 required',
    'Split strategy':        'Hold out ONE full camera site for test (no leakage)',
    'Target annotation size':'500–1,000 frames minimum (daytime + dawn/dusk)',
    'Nighttime strategy':    'Apply synthetic augmentation to annotated daytime frames',
    'Quality control':       'CVAT review pass after Roboflow annotation',
    'Status':                'Footage access pending WWF Pakistan collaboration',
}

print('=== WWF PTZ Custom Dataset — Annotation Plan ===')
for k, v in wwf_plan.items():
    if isinstance(v, dict):
        print(f'  {k}:')
        for cls_id, desc in v.items():
            print(f'    Class {cls_id}: {desc}')
    else:
        print(f'  {k:<35}: {v}')

In [ ]:
# ── WWF Dataset: Class rationale — why 3 classes not 2 ───────────────────────
print('=== Why We Annotate Clouds as a Third Class ===')
print()
rationale = [
    ('Problem',   'PTZ cameras face the horizon. Clouds, morning mist, and smoke'),
    ('',          'all appear in the same frame region at the same altitude.'),
    ('Evidence',  'benchmarking2024 shows models without explicit cloud hard negatives'),
    ('',          'produce 30-40% higher false alarm rates in deployment.'),
    ('Solution',  'Annotate cloud/mist regions explicitly as class 2 (cloud).'),
    ('Effect',    'Model learns to distinguish cloud texture/motion from smoke,'),
    ('',          'rather than treating cloud-free images as the only negative class.'),
    ('CLIP role', 'Low-confidence detections (0.3-0.6) are routed to CLIP with prompts:'),
    ('',          '"smoke rising from burning trees" vs "cloud drifting over mountain".'),
]
for k, v in rationale:
    prefix = f'  [{k}]' if k else '       '
    print(f'  {prefix:<10} {v}')

In [ ]:
# ── WWF Dataset: Projected split strategy ────────────────────────────────────
wwf_split_plan = pd.DataFrame([
    {'Split': 'Train', 'Camera sites': 'Sites A, B, C',  'Approx frames': '~600',
     'Conditions': 'Daytime + synthetic night', 'Notes': 'Multi-condition augmentation'},
    {'Split': 'Val',   'Camera sites': 'Site D',         'Approx frames': '~150',
     'Conditions': 'Daytime',                 'Notes': 'Tune thresholds'},
    {'Split': 'Test',  'Camera sites': 'Site E (held out)', 'Approx frames': '~250',
     'Conditions': 'Daytime + dawn/dusk',     'Notes': 'Geographic leakage prevention'},
])
print('WWF PTZ Dataset — Projected Split Strategy')
wwf_split_plan.set_index('Split')

---
## 5. Cross-Dataset Analysis

This section examines how the datasets complement each other and what each contributes to the three-gap problem identified in the SOA.

In [ ]:
# ── Gap coverage matrix ───────────────────────────────────────────────────────
gap_matrix = pd.DataFrame([
    {'Dataset': 'D-Fire',          'Edge benchmark': '✓', 'Cloud hard neg.': '✗', 'Nighttime': '✗', 'Fixed camera': '✗', 'S. Asian terrain': '✗'},
    {'Dataset': 'FIgLib',          'Edge benchmark': '~', 'Cloud hard neg.': '~', 'Nighttime': '✗', 'Fixed camera': '✓', 'S. Asian terrain': '✗'},
    {'Dataset': 'SKLFS-WildFire',  'Edge benchmark': '✗', 'Cloud hard neg.': '~', 'Nighttime': '✗', 'Fixed camera': '✗', 'S. Asian terrain': '✗'},
    {'Dataset': 'FASDD',           'Edge benchmark': '~', 'Cloud hard neg.': '✗', 'Nighttime': '✗', 'Fixed camera': '✗', 'S. Asian terrain': '~'},
    {'Dataset': 'VisiFire',        'Edge benchmark': '✗', 'Cloud hard neg.': '✗', 'Nighttime': '✗', 'Fixed camera': '✗', 'S. Asian terrain': '✗'},
    {'Dataset': 'FIRESENSE',       'Edge benchmark': '✗', 'Cloud hard neg.': '✗', 'Nighttime': '✗', 'Fixed camera': '✗', 'S. Asian terrain': '✗'},
    {'Dataset': 'FLAME 2',         'Edge benchmark': '✗', 'Cloud hard neg.': '✗', 'Nighttime': '✓', 'Fixed camera': '✗', 'S. Asian terrain': '✗'},
    {'Dataset': 'WWF (Custom)',     'Edge benchmark': '✓', 'Cloud hard neg.': '✓', 'Nighttime': '✓', 'Fixed camera': '✓', 'S. Asian terrain': '✓'},
])

gap_matrix.set_index('Dataset', inplace=True)
print('Gap Coverage Matrix   ✓=addressed  ~=partial  ✗=not addressed')
gap_matrix

In [ ]:
# ── Three-stage pipeline visualisation ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.axis('off')

stages = [
    ('Stage 1\nPretraining',
     'D-Fire + FIgLib\n+ SKLFS + FASDD\n+ VisiFire + FIRESENSE',
     '#2980b9', 0.05),
    ('Stage 2\nNighttime\nFine-tuning',
     'FLAME 2\n(RGB + IR)\n+ Synthetic Night Aug.',
     '#8e44ad', 0.40),
    ('Stage 3\nDomain\nAdaptation',
     'WWF PTZ\nCustom Dataset\n(South Asian terrain)',
     '#27ae60', 0.73),
]

for title, data, color, x in stages:
    rect = patches.FancyBboxPatch((x, 0.1), 0.24, 0.8,
                                  boxstyle='round,pad=0.02',
                                  linewidth=2, edgecolor=color,
                                  facecolor=color + '22')
    ax.add_patch(rect)
    ax.text(x + 0.12, 0.83, title, ha='center', va='top',
            fontsize=11, fontweight='bold', color=color)
    ax.text(x + 0.12, 0.58, data,  ha='center', va='top',
            fontsize=9, color='#2c3e50', linespacing=1.5)
    if x < 0.73:
        ax.annotate('', xy=(x + 0.27, 0.5), xytext=(x + 0.245, 0.5),
                    arrowprops=dict(arrowstyle='->', color='#7f8c8d', lw=2))

ax.text(0.5, 0.02, 'Sequential staged fine-tuning — each stage inherits weights from the previous',
        ha='center', fontsize=9, style='italic', color='#7f8c8d',
        transform=ax.transAxes)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title('Three-Stage Training Pipeline — Dataset Flow', fontsize=13, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig('training_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Dataset Strategy Summary

| Aspect | Decision | Rationale |
|---|---|---|
| **Primary eval benchmark** | D-Fire | Standard compact-model benchmark; fire + smoke classes; YOLO format |
| **PTZ-analogue pretraining** | FIgLib | Only public fixed-camera forest smoke dataset |
| **Nighttime fine-tuning** | FLAME 2 | Only public RGB+IR paired wildfire dataset |
| **Generalisation test** | FASDD | Broadest geographic diversity |
| **Hard negative strategy** | Cloud class in WWF + D-Fire confounder split | SOA gap: 30–40% FP reduction with explicit hard negatives |
| **Domain gap solution** | WWF custom annotations | No public dataset covers South Asian PTZ forest footage |
| **Annotation format** | YOLO (all stages) | Direct compatibility with YOLOv8 training loop |
| **Test leakage prevention** | Hold out one full camera site | Prevent geographic inflation of test metrics |

In [ ]:
# ── Final: Dataset size comparison (log scale) ────────────────────────────────
sizes = {
    'D-Fire':        21527,
    'FIgLib':        24800,
    'FASDD':         120000,
    'FLAME 2 (clf)': 48000,
    'SKLFS':         350000,
    'VisiFire':      10000,
    'FIRESENSE':     25000,
    'WWF (TBD)':     1000,
}

fig, ax = plt.subplots(figsize=(11, 4))
colors_map = {
    'D-Fire': '#2980b9', 'FIgLib': '#2980b9', 'FASDD': '#2980b9',
    'FLAME 2 (clf)': '#8e44ad', 'SKLFS': '#2980b9',
    'VisiFire': '#2980b9', 'FIRESENSE': '#2980b9', 'WWF (TBD)': '#27ae60'
}
bar_colors = [colors_map[k] for k in sizes]
bars = ax.barh(list(sizes.keys()), list(sizes.values()),
               color=bar_colors, edgecolor='black', linewidth=0.6)
ax.set_xscale('log')
ax.set_xlabel('Number of Images (log scale)')
ax.set_title('Dataset Sizes Across All Stages', fontweight='bold')
for bar, val in zip(bars, sizes.values()):
    ax.text(val * 1.05, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=8)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2980b9', label='Stage 1 — Pretraining'),
    Patch(facecolor='#8e44ad', label='Stage 2 — Nighttime Fine-tuning'),
    Patch(facecolor='#27ae60', label='Stage 3 — Domain Adaptation (WWF)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=8)
plt.tight_layout()
plt.savefig('dataset_sizes.png', dpi=150, bbox_inches='tight')
plt.show()